# Build: `adif__mainDataTable_notebook`

**Purpose:** Rebuild the ADIF combined table from the updated-FPD view, then append normalized social (Meta / TikTok) rows mapped to ADIF-compatible columns.

**Target table:** `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`

**Key sources:**
- `looker-studio-pro-452620.repo_stg.adif__prisma_expanded_plus_dcm_updated_fpd_view` — base ADIF view
- `looker-studio-pro-452620.repo_stg.stg__adif__social_crossplatform` — social daily facts
- `looker-studio-pro-452620.repo_int.crossplatform_pacing` — WP campaign pacing budgets

**Sections:**
1. Table Rebuild — `CREATE OR REPLACE TABLE` from the updated-FPD ADIF view
2. Social Append — normalize & append WP social rows via `INSERT INTO`

---
> **How to run in BigQuery Studio:** Open this notebook in BigQuery Studio, confirm the project is set to `looker-studio-pro-452620`, then run cells top to bottom.

## Setup

In [ ]:
# @title Project & region config
PROJECT_ID = "looker-studio-pro-452620"  # @param {type:"string"}
REGION     = "US"                         # @param {type:"string"}

print(f"Project : {PROJECT_ID}")
print(f"Region  : {REGION}")

In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

---
## SECTION 1 — Table Rebuild

Drop and recreate the target table from the updated-FPD ADIF view.
This captures the current schema and all base (non-social) rows.

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- @title: Build adif__mainDataTable_notebook — Section 1: Table Rebuild
-- @description: Recreate the target table from the updated-FPD ADIF view (schema + base rows).
-- @inputs:  looker-studio-pro-452620.repo_stg.adif__prisma_expanded_plus_dcm_updated_fpd_view
-- @outputs: looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook
-- @last_updated: 2026-02-18

CREATE OR REPLACE TABLE `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook` AS
SELECT
  *
FROM `looker-studio-pro-452620.repo_stg.adif__prisma_expanded_plus_dcm_updated_fpd_view`;

### Quick row-count check after rebuild

In [ ]:
%%bigquery --project looker-studio-pro-452620

SELECT
  COUNT(*) AS row_count,
  MIN(date) AS min_date,
  MAX(date) AS max_date
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`;

---
## SECTION 2 — Social Append

Append WP social rows (Meta / TikTok) with:
- **ad_set → package** mapping
- **ad → placement** mapping

**Sub-steps (CTEs):**

| CTE | Purpose |
|-----|---------|
| `social_daily` | Normalize social source to ad-level daily grain |
| `social_tokens` | Parse ad_set/ad naming tokens for dimension mapping |
| `social_enriched` | Resolve mapped dimensions + ad-group daily helpers |
| `pacing_dedup` | Keep WP campaign pacing rows, deduped at adgroup flight-window level |
| `pacing_daily` | Convert pacing budgets → daily planned spend |
| `social_joined` | Join social facts to pacing plan |
| `social_mapped` | Map social rows into ADIF-compatible fields |
| `social_with_rollups` | Compute package-level totals for pacing / over-under flags |
| Final `SELECT` | Output social rows aligned to target table columns |

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- @title: Build adif__mainDataTable_notebook — Section 2: Social Append
-- @description: Append WP social rows with ad_set->package and ad->placement mapping.
-- @inputs:
--   looker-studio-pro-452620.repo_stg.stg__adif__social_crossplatform
--   looker-studio-pro-452620.repo_int.crossplatform_pacing
-- @outputs: looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook
-- @last_updated: 2026-02-18

INSERT INTO `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook` (
  package_id_joined,
  date,
  flight_status_flag,
  d_daily_recalculated_cost,
  d_daily_recalculated_imps,
  d_clicks,
  d_video_plays,
  d_video_comps,
  d_min_date,
  d_max_date,
  d_min_flight_date,
  d_max_flight_date,
  d_daily_cpm,
  d_total_delivered_imps,
  d_total_del_inflight_imps,
  planned_daily_spend_pk,
  planned_daily_impressions_pk,
  final_spend,
  final_impressions,
  final_clicks,
  data_source_primary,
  pkg_est_spend,
  pkg_act_spend,
  pkg_est_imps,
  pkg_act_imps,
  pkg_over_bool,
  pkg_over_flag,
  package_type,
  cost_method,
  buy_type,
  buy_category,
  advertiser_name,
  campaign_name,
  supplier_code,
  supplier_name,
  planned_clicks,
  planned_impressions,
  channel_if_buy_category_custom_45,
  package_id,
  package_name,
  PlacementName,
  placement_name,
  placement_id,
  package_group_name,
  start_date,
  end_date,
  report_date,
  script_run_date,
  channel,
  channel_raw,
  channel_group,
  campaign_friendly,
  p_package_friendly,
  placement_type_site,
  line_item_name,
  media_type,
  funnel,
  kpi,
  initiative,
  initative,
  n_of_placements,
  planned_imps_pk,
  planned_cost_pk
)
WITH

-- * [2.1] SOCIAL DAILY: Normalize social source to ad-level daily grain.
  social_daily AS (
    SELECT
      date_day,
      CASE
        WHEN LOWER(social_platform) IN ('facebook_ads', 'instagram_ads', 'meta') THEN 'meta'
        WHEN LOWER(social_platform) IN ('tiktok_ads', 'tiktok_ads_adif', 'tiktok') THEN 'tiktok'
        ELSE LOWER(social_platform)
      END AS social_platform_norm,
      CAST(campaign_id AS STRING) AS campaign_id,
      campaign_name,
      CAST(ad_group_id AS STRING) AS ad_group_id,
      ad_group_name,
      CAST(ad_id AS STRING) AS ad_id,
      ad_name,
      account_name,
      SUM(spend)              AS spend,
      SUM(impressions)        AS impressions,
      SUM(clicks)             AS clicks,
      SUM(video_play)         AS video_play,
      SUM(video_view)         AS video_view,
      SUM(video_views_p_100)  AS video_complete_proxy
    FROM `looker-studio-pro-452620.repo_stg.stg__adif__social_crossplatform`
    GROUP BY 1,2,3,4,5,6,7,8,9
  ),

-- * [2.2] TOKEN PREP: Parse ad_set/ad naming tokens for dimension mapping.
  social_tokens AS (
    SELECT
      s.*,
      SPLIT(s.ad_group_name, '_')[SAFE_OFFSET(1)] AS initiative_token_ag,
      SPLIT(s.ad_group_name, '_')[SAFE_OFFSET(2)] AS funnel_token_ag,
      LOWER(
        REGEXP_EXTRACT(
          s.ad_group_name,
          r'_(facebook|instagram|tiktok|youtube|snapchat|linkedin|pinterest)(?:_|$)'
        )
      ) AS supplier_token_ag,
      SPLIT(s.ad_name, '_')[SAFE_OFFSET(1)] AS initiative_token_ad,
      SPLIT(s.ad_name, '_')[SAFE_OFFSET(2)] AS line_item_name_token_ad,
      SPLIT(s.ad_name, '_')[SAFE_OFFSET(3)] AS placement_type_token_1_ad,
      SPLIT(s.ad_name, '_')[SAFE_OFFSET(4)] AS buy_category_token_1_ad
    FROM social_daily AS s
  ),

-- * [2.3] SOCIAL ENRICHMENT: Resolve mapped dimensions + ad-group daily allocation helpers.
  social_enriched AS (
    SELECT
      t.*,
      COALESCE(t.initiative_token_ag, t.initiative_token_ad, 'NA')  AS initiative,
      COALESCE(t.line_item_name_token_ad, 'NA')                      AS line_item_name,
      CONCAT(
        COALESCE(t.placement_type_token_1_ad, 'na'),
        '_',
        COALESCE(t.buy_category_token_1_ad, 'na')
      ) AS placement_type_buy_category,
      CASE
        WHEN t.supplier_token_ag = 'facebook'              THEN 'FB'
        WHEN t.supplier_token_ag = 'instagram'             THEN 'IG'
        WHEN t.supplier_token_ag = 'tiktok'                THEN 'TT'
        WHEN t.social_platform_norm = 'meta'               THEN 'FB'
        WHEN t.social_platform_norm = 'tiktok'             THEN 'TT'
        ELSE UPPER(SUBSTR(t.social_platform_norm, 1, 2))
      END AS supplier_code_short,
      CASE
        WHEN t.supplier_token_ag = 'facebook'              THEN 'Facebook'
        WHEN t.supplier_token_ag = 'instagram'             THEN 'Instagram'
        WHEN t.supplier_token_ag = 'tiktok'                THEN 'TikTok'
        WHEN t.social_platform_norm = 'meta'               THEN 'Facebook'
        WHEN t.social_platform_norm = 'tiktok'             THEN 'TikTok'
        ELSE INITCAP(t.social_platform_norm)
      END AS supplier_name_full,
      SUM(t.spend) OVER (
        PARTITION BY t.date_day, t.social_platform_norm, t.campaign_id, t.ad_group_id
      ) AS ad_group_day_spend,
      COUNT(*) OVER (
        PARTITION BY t.date_day, t.social_platform_norm, t.campaign_id, t.ad_group_id
      ) AS ad_group_day_ad_count
    FROM social_tokens AS t
  ),

-- * [2.4] PACING DEDUPE: Keep WP campaign pacing rows, deduped at adgroup flight-window level.
  pacing_dedup AS (
    SELECT
      CASE
        WHEN LOWER(platform) IN ('facebook', 'instagram', 'meta') THEN 'meta'
        WHEN LOWER(platform) = 'tiktok'                           THEN 'tiktok'
        ELSE LOWER(platform)
      END AS social_platform_norm,
      CAST(c_id  AS STRING) AS campaign_id,
      CAST(ag_id AS STRING) AS ad_group_id,
      ag_start_date,
      ag_end_date,
      MAX(final_budget) AS final_budget
    FROM `looker-studio-pro-452620.repo_int.crossplatform_pacing`
    WHERE REGEXP_CONTAINS(IFNULL(campaign_name, ''), r'WP_')
      AND final_budget   > 0
      AND ag_start_date IS NOT NULL
      AND ag_end_date   IS NOT NULL
    GROUP BY 1,2,3,4,5
  ),

-- * [2.5] PACING DAILY: Convert pacing budgets → daily planned spend per campaign/platform/ad_set.
  pacing_daily AS (
    SELECT
      day AS date_day,
      social_platform_norm,
      campaign_id,
      ad_group_id,
      SUM(
        final_budget / NULLIF(DATE_DIFF(ag_end_date, ag_start_date, DAY) + 1, 0)
      ) AS planned_daily_spend
    FROM pacing_dedup,
      UNNEST(GENERATE_DATE_ARRAY(ag_start_date, ag_end_date)) AS day
    GROUP BY 1,2,3,4
  ),

-- * [2.6] SOCIAL + PLAN JOIN: Join social facts to pacing plan by date + platform + campaign + ad_set.
  social_joined AS (
    SELECT
      s.*,
      p.planned_daily_spend
    FROM social_enriched AS s
    LEFT JOIN pacing_daily AS p
      ON  s.date_day            = p.date_day
      AND s.social_platform_norm = p.social_platform_norm
      AND s.campaign_id          = p.campaign_id
      AND s.ad_group_id          = p.ad_group_id
  ),

-- * [2.7] BASE MAPPING: Map social rows into ADIF-compatible fields.
  social_mapped AS (
    SELECT
      CONCAT('Social_', supplier_code_short, '_', initiative, '_', line_item_name) AS package_id_joined,
      date_day                                                                       AS date,
      CAST(ad_group_id AS STRING)                                                    AS package_id,
      ad_group_name                                                                  AS package_name,
      ad_name                                                                        AS placement_name,
      CAST(ad_id AS STRING)                                                          AS placement_id,
      ad_name                                                                        AS PlacementName,
      ad_group_name                                                                  AS package_group_name,
      CONCAT('Social_', supplier_code_short, '_', initiative, '_', line_item_name)  AS p_package_friendly,
      placement_type_buy_category                                                    AS placement_type_site,
      placement_type_buy_category                                                    AS buy_category,
      'social'                                                                       AS buy_type,
      CONCAT('social_', LOWER(placement_type_buy_category))                          AS channel_if_buy_category_custom_45,
      CASE
        WHEN COALESCE(video_play, 0) > 0 OR COALESCE(video_complete_proxy, 0) > 0   THEN 'Video'
        WHEN REGEXP_CONTAINS(ad_group_name, r'(?i)_VV(_|$)|_Video(_|$)|_Reel(_|$)') THEN 'Video'
        WHEN REGEXP_CONTAINS(ad_name, r'(?i)video|reel')                             THEN 'Video'
        ELSE 'Static'
      END AS media_type,
      CASE
        WHEN funnel_token_ag IN ('VV', 'Reach') THEN 'Awareness'
        WHEN funnel_token_ag = 'Engagement'      THEN 'Consideration'
        ELSE 'Awareness'
      END AS funnel,
      CASE
        WHEN funnel_token_ag IN ('VV', 'VideoViews', 'Video') THEN 'Video Views'
        WHEN funnel_token_ag = 'Reach'                        THEN 'Impressions'
        WHEN funnel_token_ag = 'Engagement'                   THEN 'Engagement'
        ELSE 'Impressions'
      END AS kpi,
      initiative,
      initiative                                            AS initative,
      line_item_name,
      spend                                                 AS final_spend,
      CAST(impressions AS FLOAT64)                          AS final_impressions,
      CAST(clicks AS FLOAT64)                               AS final_clicks,
      spend                                                 AS d_daily_recalculated_cost,
      CAST(impressions AS INT64)                            AS d_daily_recalculated_imps,
      CAST(clicks AS INT64)                                 AS d_clicks,
      SAFE_CAST(ROUND(video_play) AS INT64)                 AS d_video_plays,
      SAFE_CAST(ROUND(video_complete_proxy) AS INT64)       AS d_video_comps,
      CASE
        WHEN planned_daily_spend IS NULL          THEN NULL
        WHEN ad_group_day_spend > 0               THEN planned_daily_spend * SAFE_DIVIDE(spend, ad_group_day_spend)
        WHEN ad_group_day_ad_count > 0            THEN planned_daily_spend / ad_group_day_ad_count
        ELSE NULL
      END AS planned_daily_spend_pk,
      CAST(NULL AS FLOAT64)                                 AS planned_daily_impressions_pk,
      supplier_name_full                                    AS supplier_name,
      supplier_code_short                                   AS supplier_code,
      campaign_name,
      social_platform_norm,
      campaign_id,
      ad_group_id
    FROM social_joined
  ),

-- * [2.8] ROLLUPS: Compute package-level totals for pacing and over/under flags.
  social_with_rollups AS (
    SELECT
      *,
      MIN(date) OVER (PARTITION BY package_id)               AS start_date_pkg,
      MAX(date) OVER (PARTITION BY package_id)               AS end_date_pkg,
      SUM(final_spend) OVER (PARTITION BY package_id)        AS pkg_act_spend,
      SUM(final_impressions) OVER (PARTITION BY package_id)  AS pkg_act_imps,
      SUM(planned_daily_spend_pk) OVER (PARTITION BY package_id) AS pkg_est_spend
    FROM social_mapped
  )

-- * [2.9] FINAL SOCIAL ROWS: Output aligned to target table columns.
SELECT
  package_id_joined,
  date,
  CASE
    WHEN CURRENT_DATE() > end_date_pkg THEN 'ended'
    ELSE 'live'
  END                                                          AS flight_status_flag,
  d_daily_recalculated_cost,
  d_daily_recalculated_imps,
  d_clicks,
  d_video_plays,
  d_video_comps,
  start_date_pkg                                               AS d_min_date,
  end_date_pkg                                                 AS d_max_date,
  start_date_pkg                                               AS d_min_flight_date,
  end_date_pkg                                                 AS d_max_flight_date,
  CASE
    WHEN final_impressions > 0 THEN (final_spend * 1000) / final_impressions
    ELSE NULL
  END                                                          AS d_daily_cpm,
  SAFE_CAST(ROUND(pkg_act_imps) AS INT64)                      AS d_total_delivered_imps,
  SAFE_CAST(ROUND(pkg_act_imps) AS INT64)                      AS d_total_del_inflight_imps,
  planned_daily_spend_pk,
  planned_daily_impressions_pk,
  final_spend,
  final_impressions,
  final_clicks,
  CASE
    WHEN social_platform_norm = 'meta' THEN 'meta'
    WHEN social_platform_norm = 'tiktok' THEN 'tiktok'
    ELSE 'social'
  END                                                          AS data_source_primary,
  pkg_est_spend,
  pkg_act_spend,
  CAST(NULL AS FLOAT64)                                        AS pkg_est_imps,
  pkg_act_imps,
  CASE
    WHEN pkg_est_spend IS NULL OR pkg_est_spend = 0 THEN NULL
    ELSE pkg_act_spend > pkg_est_spend
  END                                                          AS pkg_over_bool,
  CASE
    WHEN pkg_est_spend IS NULL OR pkg_est_spend = 0 THEN NULL
    WHEN pkg_act_spend > pkg_est_spend               THEN 1
    ELSE 0
  END                                                          AS pkg_over_flag,
  'Standalone'                                                 AS package_type,
  CAST(NULL AS STRING)                                         AS cost_method,
  buy_type,
  buy_category,
  'Forevermark US'                                             AS advertiser_name,
  campaign_name,
  supplier_code,
  supplier_name,
  CAST(NULL AS INT64)                                          AS planned_clicks,
  CAST(NULL AS INT64)                                          AS planned_impressions,
  channel_if_buy_category_custom_45,
  package_id,
  package_name,
  PlacementName,
  placement_name,
  placement_id,
  package_group_name,
  start_date_pkg                                               AS start_date,
  end_date_pkg                                                 AS end_date,
  CURRENT_DATE()                                               AS report_date,
  CURRENT_TIMESTAMP()                                          AS script_run_date,
  CASE
    WHEN media_type = 'Video' THEN 'social_paid video'
    ELSE 'social_paid display'
  END                                                          AS channel,
  'social'                                                     AS channel_raw,
  'social'                                                     AS channel_group,
  campaign_name                                                AS campaign_friendly,
  p_package_friendly,
  placement_type_site,
  line_item_name,
  media_type,
  funnel,
  kpi,
  initiative,
  initative,
  1                                                            AS n_of_placements,
  CAST(NULL AS INT64)                                          AS planned_imps_pk,
  pkg_est_spend                                                AS planned_cost_pk
FROM social_with_rollups;

---
## Verification

Confirm total row counts and a social vs. non-social split after both sections run.

In [ ]:
%%bigquery --project looker-studio-pro-452620

SELECT

  COUNT(*)           AS row_count,
  MIN(date)          AS min_date,
  MAX(date)          AS max_date,
  ROUND(SUM(final_spend), 2)        AS total_spend,
  ROUND(SUM(final_impressions), 0)  AS total_impressions
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`
WHERE EXTRACT(YEAR FROM date) = 2026



In [ ]:
%%bigquery --project looker-studio-pro-452620

SELECT
  data_source_primary,
  COUNT(*)           AS row_count,
  MIN(date)          AS min_date,
  MAX(date)          AS max_date,
  ROUND(SUM(final_spend), 2)        AS total_spend,
  ROUND(SUM(final_impressions), 0)  AS total_impressions
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`
WHERE EXTRACT(YEAR FROM date) = 2026
GROUP BY 1
ORDER BY 1;

In [ ]:
# dataframe: df
# uuid: 87F30F4B-8F8B-40EB-B34E-DC6861A726A6
# output_variable:
# config_str: CqUkeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbXSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJkYXRlUmFuZ2VEaW1lbnNpb24iOiJxdF8zdWNoOHJoMjBkIiwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfa21sMHBzaDIwZCJdfX1dfSwibWV0cmljcyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfNXVjaDhyaDIwZCIsInF0XzJzdXB0c2gyMGQiXX19XX0sInBpdm90VGFibGVQcm9wZXJ0eSI6eyJwaXZvdFJvbGx1cENvbmZpZyI6eyJlbmFibGVSb3dSb2xsdXAiOmZhbHNlLCJlbmFibGVDb2xSb2xsdXAiOmZhbHNlfSwiY2VsbFdpZHRoIjpbMTAwLDEwMF0sInJvd0xhYmVsV2lkdGgiOltdLCJncmFuZFRvdGFsV2lkdGgiOlsxMDBdLCJ0YWJsZU1ldHJpY1Byb3BlcnR5IjpbXSwicGl2b3REaXNwbGF5Q29uZmlnIjp7InJvdyI6W3sic29ydCI6W3sic29ydERpciI6MSwic29ydENvbHVtbiI6InF0XzV1Y2g4cmgyMGQifV19XSwiY29sIjpbXSwiaGllcmFyY2h5IjpbeyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfNXVjaDhyaDIwZCJ9XX0seyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfNXVjaDhyaDIwZCJ9XX1dfSwiYmFja2dyb3VuZEFuZEJvcmRlclByb3BlcnR5Ijp7ImJvcmRlciI6eyJvcGFjaXR5IjowLCJzaXplIjowLCJyYWRpdXMiOjB9fSwiY2hhcnRUaXRsZVByb3BlcnR5Ijp7ImNoYXJ0VGl0bGVBdXRvR2VuZXJhdGVUZXh0Ijp0cnVlLCJjaGFydFRpdGxlVGV4dCI6IlRhYmxlIn19LCJjb21wb25lbnRQcm9wZXJ0eU1pZ3JhdGlvblN0YXR1cyI6Mn19LCJjb25jZXB0RGVmcyI6W3siaWQiOiJ0MC5xdF81dWNoOHJoMjBkIiwibmFtZSI6InF0XzV1Y2g4cmgyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJ0b3RhbF9pbXByZXNzaW9ucyIsImFnZ3JlZ2F0aW9uIjo2fX19LHsiaWQiOiJ0MC5xdF8yc3VwdHNoMjBkIiwibmFtZSI6InF0XzJzdXB0c2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJ0b3RhbF9zcGVuZCIsImFnZ3JlZ2F0aW9uIjo2fX19LHsiaWQiOiJ0MC5xdF9rbWwwcHNoMjBkIiwibmFtZSI6InF0X2ttbDBwc2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19fSx7ImlkIjoidDAucXRfM3VjaDhyaDIwZCIsIm5hbWUiOiJxdF8zdWNoOHJoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoibWluX2RhdGUifX19LHsiaWQiOiJ0MC5xdF81bXIxajFoMjBkIiwibmFtZSI6InF0XzVtcjFqMWgyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJtYXhfZGF0ZSJ9fX1dLCJhdHRyaWJ1dGVDb25maWciOnsiY29tcG9uZW50QXR0cmlidXRlIjp7ImRpc3BsYXlDb25maWdWZXJzaW9uIjowLCJkYXRhc291cmNlQ29uZmlnVmVyc2lvbiI6MiwidG9wIjowLCJsZWZ0IjowLCJ3aWR0aCI6ODIzLCJoZWlnaHQiOjU3MH19LCJjb21wb25lbnRJZCI6Il9fVklaX0NIQVJUX0lEX18iLCJ0eXBlIjoicGl2b3QtdGFibGUiLCJwcmVzZXQiOiJkZWZhdWx0IiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6Im9uRGltZW5zaW9uRXhwYW5kQ29sbGFwc2UiLCJ2YWx1ZSI6eyJhcnJheVZhbHVlIjp7InZhbHVlIjpbeyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiYWN0aW9uIiwidmFsdWUiOnsic3RyVmFsdWUiOiJleHBhbmQtY29sbGFwc2UifX0seyJrZXkiOiJpc0NvbnRyb2wiLCJ2YWx1ZSI6eyJib29sVmFsdWUiOmZhbHNlfX0seyJrZXkiOiJyZXNvdXJjZSIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJ0eXBlIiwidmFsdWUiOnsic3RyVmFsdWUiOiJjb25jZXB0cyJ9fSx7ImtleSI6InZhbHVlIiwidmFsdWUiOnsiYXJyYXlWYWx1ZSI6eyJ2YWx1ZSI6W3sibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6ImlkIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MC5xdF9rbWwwcHNoMjBkIn19LHsia2V5IjoibmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicXRfa21sMHBzaDIwZCJ9fSx7ImtleSI6Im5hbWVzcGFjZSIsInZhbHVlIjp7InN0clZhbHVlIjoidDAifX0seyJrZXkiOiJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJkYXRhVHJhbnNmb3JtYXRpb24iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5Ijoic291cmNlRmllbGROYW1lIiwidmFsdWUiOnsic3RyVmFsdWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19XX19fV19fX1dfX0seyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiaWQiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InQwLnF0XzVtcjFqMWgyMGQifX0seyJrZXkiOiJuYW1lIiwidmFsdWUiOnsic3RyVmFsdWUiOiJxdF81bXIxajFoMjBkIn19LHsia2V5IjoibmFtZXNwYWNlIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6InF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6Im1heF9kYXRlIn19XX19fV19fX1dfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOlt7ImJlaGF2aW9yVHlwZSI6Im9uRGltZW5zaW9uRXhwYW5kQ29sbGFwc2UiLCJkcmlsbGRvd25QYXJhbWV0ZXJWYWx1ZSI6eyJ2YWx1ZXMiOlt7ImluaXQiOnsiaWQiOiJ0MC5xdF9rbWwwcHNoMjBkIiwibmFtZSI6InF0X2ttbDBwc2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19fSwiZGlyZWN0aW9uIjowLCJjdXIiOnsiaWQiOiJ0MC5xdF81bXIxajFoMjBkIiwibmFtZSI6InF0XzVtcjFqMWgyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsImRpc3BsYXlOYW1lIjoibWF4X2RhdGUiLCJzZW1hbnRpYyI6WzJdLCJsb29rZXJQcm9wZXJ0aWVzIjp7ImRlc2NyaXB0aW9uIjoiIn0sInR5cGUiOjAsImRhdGFUeXBlIjo2LCJpc0RlZmF1bHQiOmZhbHNlLCJkZWZhdWx0QWdncmVnYXRpb24iOjAsIm5ld0RlZmF1bHRBZ2dyZWdhdGlvbiI6MCwiaXNBdXRvRmllbGQiOnRydWUsInNlbWFudGljT3B0aW9ucyI6eyJ1c2VOYXRpdmVEYXRlVGltZSI6dHJ1ZX0sInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJtYXhfZGF0ZSJ9fSwiZ3JvdXAiOiJDaGFydCBmaWVsZHMifSwiZHJpbGxUbyI6eyJpZCI6InQwLnF0X2ttbDBwc2gyMGQiLCJuYW1lIjoicXRfa21sMHBzaDIwZCIsIm5hbWVzcGFjZSI6InQwIiwiZGlzcGxheU5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5Iiwic2VtYW50aWMiOlszMl0sImxvb2tlclByb3BlcnRpZXMiOnsiZGVzY3JpcHRpb24iOiIifSwidHlwZSI6MCwiZGF0YVR5cGUiOjAsImlzRGVmYXVsdCI6ZmFsc2UsImRlZmF1bHRBZ2dyZWdhdGlvbiI6MCwibmV3RGVmYXVsdEFnZ3JlZ2F0aW9uIjowLCJpc0F1dG9GaWVsZCI6dHJ1ZSwicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iOnsiZGF0YVRyYW5zZm9ybWF0aW9uIjp7InNvdXJjZUZpZWxkTmFtZSI6ImRhdGFfc291cmNlX3ByaW1hcnkifX0sImdyb3VwIjoiQ2hhcnQgZmllbGRzIn0sInR5cGUiOjMsImhpZXJhcmNoeSI6W3siaWQiOiJ0MC5xdF9rbWwwcHNoMjBkIiwibmFtZSI6InF0X2ttbDBwc2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19fSx7ImlkIjoidDAucXRfNW1yMWoxaDIwZCIsIm5hbWUiOiJxdF81bXIxajFoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoibWF4X2RhdGUifX19XX1dfX1dLCJ2ZXJzaW9uIjoxfRoPCgt0b3RhbF9zcGVuZBAFGhcKE2RhdGFfc291cmNlX3ByaW1hcnkQARoVChF0b3RhbF9pbXByZXNzaW9ucxAF

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='df', uuid='87F30F4B-8F8B-40EB-B34E-DC6861A726A6', config_str='CqUkeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbXSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJkYXRlUmFuZ2VEaW1lbnNpb24iOiJxdF8zdWNoOHJoMjBkIiwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfa21sMHBzaDIwZCJdfX1dfSwibWV0cmljcyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfNXVjaDhyaDIwZCIsInF0XzJzdXB0c2gyMGQiXX19XX0sInBpdm90VGFibGVQcm9wZXJ0eSI6eyJwaXZvdFJvbGx1cENvbmZpZyI6eyJlbmFibGVSb3dSb2xsdXAiOmZhbHNlLCJlbmFibGVDb2xSb2xsdXAiOmZhbHNlfSwiY2VsbFdpZHRoIjpbMTAwLDEwMF0sInJvd0xhYmVsV2lkdGgiOltdLCJncmFuZFRvdGFsV2lkdGgiOlsxMDBdLCJ0YWJsZU1ldHJpY1Byb3BlcnR5IjpbXSwicGl2b3REaXNwbGF5Q29uZmlnIjp7InJvdyI6W3sic29ydCI6W3sic29ydERpciI6MSwic29ydENvbHVtbiI6InF0XzV1Y2g4cmgyMGQifV19XSwiY29sIjpbXSwiaGllcmFyY2h5IjpbeyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfNXVjaDhyaDIwZCJ9XX0seyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfNXVjaDhyaDIwZCJ9XX1dfSwiYmFja2dyb3VuZEFuZEJvcmRlclByb3BlcnR5Ijp7ImJvcmRlciI6eyJvcGFjaXR5IjowLCJzaXplIjowLCJyYWRpdXMiOjB9fSwiY2hhcnRUaXRsZVByb3BlcnR5Ijp7ImNoYXJ0VGl0bGVBdXRvR2VuZXJhdGVUZXh0Ijp0cnVlLCJjaGFydFRpdGxlVGV4dCI6IlRhYmxlIn19LCJjb21wb25lbnRQcm9wZXJ0eU1pZ3JhdGlvblN0YXR1cyI6Mn19LCJjb25jZXB0RGVmcyI6W3siaWQiOiJ0MC5xdF81dWNoOHJoMjBkIiwibmFtZSI6InF0XzV1Y2g4cmgyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJ0b3RhbF9pbXByZXNzaW9ucyIsImFnZ3JlZ2F0aW9uIjo2fX19LHsiaWQiOiJ0MC5xdF8yc3VwdHNoMjBkIiwibmFtZSI6InF0XzJzdXB0c2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJ0b3RhbF9zcGVuZCIsImFnZ3JlZ2F0aW9uIjo2fX19LHsiaWQiOiJ0MC5xdF9rbWwwcHNoMjBkIiwibmFtZSI6InF0X2ttbDBwc2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19fSx7ImlkIjoidDAucXRfM3VjaDhyaDIwZCIsIm5hbWUiOiJxdF8zdWNoOHJoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoibWluX2RhdGUifX19LHsiaWQiOiJ0MC5xdF81bXIxajFoMjBkIiwibmFtZSI6InF0XzVtcjFqMWgyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJtYXhfZGF0ZSJ9fX1dLCJhdHRyaWJ1dGVDb25maWciOnsiY29tcG9uZW50QXR0cmlidXRlIjp7ImRpc3BsYXlDb25maWdWZXJzaW9uIjowLCJkYXRhc291cmNlQ29uZmlnVmVyc2lvbiI6MiwidG9wIjowLCJsZWZ0IjowLCJ3aWR0aCI6ODIzLCJoZWlnaHQiOjU3MH19LCJjb21wb25lbnRJZCI6Il9fVklaX0NIQVJUX0lEX18iLCJ0eXBlIjoicGl2b3QtdGFibGUiLCJwcmVzZXQiOiJkZWZhdWx0IiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6Im9uRGltZW5zaW9uRXhwYW5kQ29sbGFwc2UiLCJ2YWx1ZSI6eyJhcnJheVZhbHVlIjp7InZhbHVlIjpbeyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiYWN0aW9uIiwidmFsdWUiOnsic3RyVmFsdWUiOiJleHBhbmQtY29sbGFwc2UifX0seyJrZXkiOiJpc0NvbnRyb2wiLCJ2YWx1ZSI6eyJib29sVmFsdWUiOmZhbHNlfX0seyJrZXkiOiJyZXNvdXJjZSIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJ0eXBlIiwidmFsdWUiOnsic3RyVmFsdWUiOiJjb25jZXB0cyJ9fSx7ImtleSI6InZhbHVlIiwidmFsdWUiOnsiYXJyYXlWYWx1ZSI6eyJ2YWx1ZSI6W3sibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6ImlkIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MC5xdF9rbWwwcHNoMjBkIn19LHsia2V5IjoibmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicXRfa21sMHBzaDIwZCJ9fSx7ImtleSI6Im5hbWVzcGFjZSIsInZhbHVlIjp7InN0clZhbHVlIjoidDAifX0seyJrZXkiOiJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJkYXRhVHJhbnNmb3JtYXRpb24iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5Ijoic291cmNlRmllbGROYW1lIiwidmFsdWUiOnsic3RyVmFsdWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19XX19fV19fX1dfX0seyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiaWQiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InQwLnF0XzVtcjFqMWgyMGQifX0seyJrZXkiOiJuYW1lIiwidmFsdWUiOnsic3RyVmFsdWUiOiJxdF81bXIxajFoMjBkIn19LHsia2V5IjoibmFtZXNwYWNlIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6InF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6Im1heF9kYXRlIn19XX19fV19fX1dfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOlt7ImJlaGF2aW9yVHlwZSI6Im9uRGltZW5zaW9uRXhwYW5kQ29sbGFwc2UiLCJkcmlsbGRvd25QYXJhbWV0ZXJWYWx1ZSI6eyJ2YWx1ZXMiOlt7ImluaXQiOnsiaWQiOiJ0MC5xdF9rbWwwcHNoMjBkIiwibmFtZSI6InF0X2ttbDBwc2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19fSwiZGlyZWN0aW9uIjowLCJjdXIiOnsiaWQiOiJ0MC5xdF81bXIxajFoMjBkIiwibmFtZSI6InF0XzVtcjFqMWgyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsImRpc3BsYXlOYW1lIjoibWF4X2RhdGUiLCJzZW1hbnRpYyI6WzJdLCJsb29rZXJQcm9wZXJ0aWVzIjp7ImRlc2NyaXB0aW9uIjoiIn0sInR5cGUiOjAsImRhdGFUeXBlIjo2LCJpc0RlZmF1bHQiOmZhbHNlLCJkZWZhdWx0QWdncmVnYXRpb24iOjAsIm5ld0RlZmF1bHRBZ2dyZWdhdGlvbiI6MCwiaXNBdXRvRmllbGQiOnRydWUsInNlbWFudGljT3B0aW9ucyI6eyJ1c2VOYXRpdmVEYXRlVGltZSI6dHJ1ZX0sInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJtYXhfZGF0ZSJ9fSwiZ3JvdXAiOiJDaGFydCBmaWVsZHMifSwiZHJpbGxUbyI6eyJpZCI6InQwLnF0X2ttbDBwc2gyMGQiLCJuYW1lIjoicXRfa21sMHBzaDIwZCIsIm5hbWVzcGFjZSI6InQwIiwiZGlzcGxheU5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5Iiwic2VtYW50aWMiOlszMl0sImxvb2tlclByb3BlcnRpZXMiOnsiZGVzY3JpcHRpb24iOiIifSwidHlwZSI6MCwiZGF0YVR5cGUiOjAsImlzRGVmYXVsdCI6ZmFsc2UsImRlZmF1bHRBZ2dyZWdhdGlvbiI6MCwibmV3RGVmYXVsdEFnZ3JlZ2F0aW9uIjowLCJpc0F1dG9GaWVsZCI6dHJ1ZSwicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iOnsiZGF0YVRyYW5zZm9ybWF0aW9uIjp7InNvdXJjZUZpZWxkTmFtZSI6ImRhdGFfc291cmNlX3ByaW1hcnkifX0sImdyb3VwIjoiQ2hhcnQgZmllbGRzIn0sInR5cGUiOjMsImhpZXJhcmNoeSI6W3siaWQiOiJ0MC5xdF9rbWwwcHNoMjBkIiwibmFtZSI6InF0X2ttbDBwc2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJkYXRhX3NvdXJjZV9wcmltYXJ5In19fSx7ImlkIjoidDAucXRfNW1yMWoxaDIwZCIsIm5hbWUiOiJxdF81bXIxajFoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoibWF4X2RhdGUifX19XX1dfX1dLCJ2ZXJzaW9uIjoxfRoPCgt0b3RhbF9zcGVuZBAFGhcKE2RhdGFfc291cmNlX3ByaW1hcnkQARoVChF0b3RhbF9pbXByZXNzaW9ucxAF')

In [ ]:
%%bigquery --project looker-studio-pro-452620

SELECT
  supplier_code,
  p_package_friendly,
  COUNT(*)           AS row_count,
  MIN(date)          AS min_date,
  MAX(date)          AS max_date,
  ROUND(SUM(planned_daily_spend_pk), 0)       AS planned_spend,
  ROUND(SUM(planned_daily_impressions_pk), 0)       AS planned_imps,
  ROUND(SUM(final_spend), 0)        AS total_spend,
  ROUND(SUM(final_impressions), 0)  AS total_impressions,
  ROUND(SUM(fpd_spend), 0)        AS fpd_spend,
  ROUND(SUM(fpd_impressions), 0)  AS fpd_impressions
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`
WHERE EXTRACT(YEAR FROM date) = 2026
GROUP BY 1,2
ORDER BY 1 asc;

In [ ]:
# prompt: generate dataFrame

import pandas as pd
import pandas_gbq

sql = """
SELECT
  supplier_code,
  p_package_friendly,
  COUNT(*)           AS row_count,
  MIN(date)          AS min_date,
  MAX(date)          AS max_date,
  ROUND(SUM(planned_daily_spend_pk), 0)       AS planned_spend,
  ROUND(SUM(planned_daily_impressions_pk), 0)       AS planned_imps,
  ROUND(SUM(final_spend), 0)        AS total_spend,
  ROUND(SUM(final_impressions), 0)  AS total_impressions,
  ROUND(SUM(fpd_spend), 0)        AS fpd_spend,
  ROUND(SUM(fpd_impressions), 0)  AS fpd_impressions
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`
  WHERE EXTRACT(YEAR FROM date) = 2026
  GROUP BY 1,2
  ORDER BY 1 asc
"""

df2 = pandas_gbq.read_gbq(sql, project_id=PROJECT_ID, dialect="standard")

In [ ]:
# dataframe: df2
# uuid: DFCDBBD3-01E8-473F-A10E-A3B2EB1A33FD
# output_variable:
# config_str: CswjeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbXSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJkYXRlUmFuZ2VEaW1lbnNpb24iOiJxdF92eWFtbjNoMjBkIiwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfNzdwcXkzaDIwZCJdfX1dfSwibWV0cmljcyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfcXcydTgzaDIwZCIsInF0X3V4cHViNGgyMGQiXX19XX0sInBpdm90VGFibGVQcm9wZXJ0eSI6eyJwaXZvdFJvbGx1cENvbmZpZyI6eyJlbmFibGVSb3dSb2xsdXAiOnRydWUsImVuYWJsZUNvbFJvbGx1cCI6ZmFsc2V9LCJjZWxsV2lkdGgiOlsxMDAsMTAwXSwicm93TGFiZWxXaWR0aCI6W10sImdyYW5kVG90YWxXaWR0aCI6W10sInRhYmxlTWV0cmljUHJvcGVydHkiOltdLCJwaXZvdERpc3BsYXlDb25maWciOnsicm93IjpbeyJzb3J0IjpbeyJzb3J0RGlyIjowLCJzb3J0Q29sdW1uIjoicXRfNzdwcXkzaDIwZCJ9XX1dLCJjb2wiOltdLCJoaWVyYXJjaHkiOlt7InNvcnQiOlt7InNvcnREaXIiOjAsInNvcnRDb2x1bW4iOiJxdF83N3BxeTNoMjBkIn1dfSx7InNvcnQiOlt7InNvcnREaXIiOjAsInNvcnRDb2x1bW4iOiJxdF90cmdrMDNoMjBkIn1dfV19LCJzaG93Um93R3JhbmRSb2xsdXAiOnRydWUsImJhY2tncm91bmRBbmRCb3JkZXJQcm9wZXJ0eSI6eyJib3JkZXIiOnsib3BhY2l0eSI6MCwic2l6ZSI6MCwicmFkaXVzIjowfX19LCJjb21wb25lbnRQcm9wZXJ0eU1pZ3JhdGlvblN0YXR1cyI6Mn19LCJjb25jZXB0RGVmcyI6W3siaWQiOiJ0MC5xdF92eWFtbjNoMjBkIiwibmFtZSI6InF0X3Z5YW1uM2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJtaW5fZGF0ZSJ9fX0seyJpZCI6InQwLnF0Xzc3cHF5M2gyMGQiLCJuYW1lIjoicXRfNzdwcXkzaDIwZCIsIm5hbWVzcGFjZSI6InQwIiwicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iOnsiZGF0YVRyYW5zZm9ybWF0aW9uIjp7InNvdXJjZUZpZWxkTmFtZSI6InN1cHBsaWVyX2NvZGUifX19LHsiaWQiOiJ0MC5xdF90cmdrMDNoMjBkIiwibmFtZSI6InF0X3RyZ2swM2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJwX3BhY2thZ2VfZnJpZW5kbHkifX19LHsiaWQiOiJ0MC5xdF9xdzJ1ODNoMjBkIiwibmFtZSI6InF0X3F3MnU4M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJwbGFubmVkX2ltcHMiLCJhZ2dyZWdhdGlvbiI6Nn19fSx7ImlkIjoidDAucXRfdXhwdWI0aDIwZCIsIm5hbWUiOiJxdF91eHB1YjRoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoidG90YWxfaW1wcmVzc2lvbnMiLCJhZ2dyZWdhdGlvbiI6Nn19fV0sImF0dHJpYnV0ZUNvbmZpZyI6eyJjb21wb25lbnRBdHRyaWJ1dGUiOnsiZGlzcGxheUNvbmZpZ1ZlcnNpb24iOjAsImRhdGFzb3VyY2VDb25maWdWZXJzaW9uIjoyLCJ0b3AiOjAsImxlZnQiOjAsIndpZHRoIjo4MjMsImhlaWdodCI6NTcwfX0sImNvbXBvbmVudElkIjoiX19WSVpfQ0hBUlRfSURfXyIsInR5cGUiOiJwaXZvdC10YWJsZSIsInByZXNldCI6ImRlZmF1bHQiLCJiZWhhdmlvciI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5Ijoib25EaW1lbnNpb25FeHBhbmRDb2xsYXBzZSIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOlt7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJhY3Rpb24iLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6ImV4cGFuZC1jb2xsYXBzZSJ9fSx7ImtleSI6ImlzQ29udHJvbCIsInZhbHVlIjp7ImJvb2xWYWx1ZSI6ZmFsc2V9fSx7ImtleSI6InJlc291cmNlIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InR5cGUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6ImNvbmNlcHRzIn19LHsia2V5IjoidmFsdWUiLCJ2YWx1ZSI6eyJhcnJheVZhbHVlIjp7InZhbHVlIjpbeyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiaWQiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InQwLnF0Xzc3cHF5M2gyMGQifX0seyJrZXkiOiJuYW1lIiwidmFsdWUiOnsic3RyVmFsdWUiOiJxdF83N3BxeTNoMjBkIn19LHsia2V5IjoibmFtZXNwYWNlIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6InF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InN1cHBsaWVyX2NvZGUifX1dfX19XX19fV19fSx7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJpZCIsInZhbHVlIjp7InN0clZhbHVlIjoidDAucXRfdHJnazAzaDIwZCJ9fSx7ImtleSI6Im5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InF0X3RyZ2swM2gyMGQifX0seyJrZXkiOiJuYW1lc3BhY2UiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InQwIn19LHsia2V5IjoicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiZGF0YVRyYW5zZm9ybWF0aW9uIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvdXJjZUZpZWxkTmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicF9wYWNrYWdlX2ZyaWVuZGx5In19XX19fV19fX1dfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOlt7ImJlaGF2aW9yVHlwZSI6Im9uRGltZW5zaW9uRXhwYW5kQ29sbGFwc2UiLCJkcmlsbGRvd25QYXJhbWV0ZXJWYWx1ZSI6eyJ2YWx1ZXMiOlt7ImluaXQiOnsiaWQiOiJ0MC5xdF83N3BxeTNoMjBkIiwibmFtZSI6InF0Xzc3cHF5M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJzdXBwbGllcl9jb2RlIn19fSwiZGlyZWN0aW9uIjoxLCJjdXIiOnsiaWQiOiJ0MC5xdF83N3BxeTNoMjBkIiwibmFtZSI6InF0Xzc3cHF5M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsImRpc3BsYXlOYW1lIjoic3VwcGxpZXJfY29kZSIsInNlbWFudGljIjpbMzJdLCJsb29rZXJQcm9wZXJ0aWVzIjp7ImRlc2NyaXB0aW9uIjoiIn0sInR5cGUiOjAsImRhdGFUeXBlIjowLCJpc0RlZmF1bHQiOmZhbHNlLCJkZWZhdWx0QWdncmVnYXRpb24iOjAsIm5ld0RlZmF1bHRBZ2dyZWdhdGlvbiI6MCwiaXNBdXRvRmllbGQiOnRydWUsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJzdXBwbGllcl9jb2RlIn19LCJncm91cCI6IkNoYXJ0IGZpZWxkcyJ9LCJkcmlsbFRvIjp7ImlkIjoidDAucXRfdHJnazAzaDIwZCIsIm5hbWUiOiJxdF90cmdrMDNoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJkaXNwbGF5TmFtZSI6InBfcGFja2FnZV9mcmllbmRseSIsInNlbWFudGljIjpbMzJdLCJsb29rZXJQcm9wZXJ0aWVzIjp7ImRlc2NyaXB0aW9uIjoiIn0sInR5cGUiOjAsImRhdGFUeXBlIjowLCJpc0RlZmF1bHQiOmZhbHNlLCJkZWZhdWx0QWdncmVnYXRpb24iOjAsIm5ld0RlZmF1bHRBZ2dyZWdhdGlvbiI6MCwiaXNBdXRvRmllbGQiOnRydWUsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJwX3BhY2thZ2VfZnJpZW5kbHkifX0sImdyb3VwIjoiQ2hhcnQgZmllbGRzIn0sInR5cGUiOjMsImhpZXJhcmNoeSI6W3siaWQiOiJ0MC5xdF83N3BxeTNoMjBkIiwibmFtZSI6InF0Xzc3cHF5M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJzdXBwbGllcl9jb2RlIn19fSx7ImlkIjoidDAucXRfdHJnazAzaDIwZCIsIm5hbWUiOiJxdF90cmdrMDNoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoicF9wYWNrYWdlX2ZyaWVuZGx5In19fV19XX19XSwidmVyc2lvbiI6MX0aEQoNc3VwcGxpZXJfY29kZRABGhUKEXRvdGFsX2ltcHJlc3Npb25zEAUaEAoMcGxhbm5lZF9pbXBzEAU=

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='df2', uuid='DFCDBBD3-01E8-473F-A10E-A3B2EB1A33FD', config_str='CswjeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbXSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJkYXRlUmFuZ2VEaW1lbnNpb24iOiJxdF92eWFtbjNoMjBkIiwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfNzdwcXkzaDIwZCJdfX1dfSwibWV0cmljcyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfcXcydTgzaDIwZCIsInF0X3V4cHViNGgyMGQiXX19XX0sInBpdm90VGFibGVQcm9wZXJ0eSI6eyJwaXZvdFJvbGx1cENvbmZpZyI6eyJlbmFibGVSb3dSb2xsdXAiOnRydWUsImVuYWJsZUNvbFJvbGx1cCI6ZmFsc2V9LCJjZWxsV2lkdGgiOlsxMDAsMTAwXSwicm93TGFiZWxXaWR0aCI6W10sImdyYW5kVG90YWxXaWR0aCI6W10sInRhYmxlTWV0cmljUHJvcGVydHkiOltdLCJwaXZvdERpc3BsYXlDb25maWciOnsicm93IjpbeyJzb3J0IjpbeyJzb3J0RGlyIjowLCJzb3J0Q29sdW1uIjoicXRfNzdwcXkzaDIwZCJ9XX1dLCJjb2wiOltdLCJoaWVyYXJjaHkiOlt7InNvcnQiOlt7InNvcnREaXIiOjAsInNvcnRDb2x1bW4iOiJxdF83N3BxeTNoMjBkIn1dfSx7InNvcnQiOlt7InNvcnREaXIiOjAsInNvcnRDb2x1bW4iOiJxdF90cmdrMDNoMjBkIn1dfV19LCJzaG93Um93R3JhbmRSb2xsdXAiOnRydWUsImJhY2tncm91bmRBbmRCb3JkZXJQcm9wZXJ0eSI6eyJib3JkZXIiOnsib3BhY2l0eSI6MCwic2l6ZSI6MCwicmFkaXVzIjowfX19LCJjb21wb25lbnRQcm9wZXJ0eU1pZ3JhdGlvblN0YXR1cyI6Mn19LCJjb25jZXB0RGVmcyI6W3siaWQiOiJ0MC5xdF92eWFtbjNoMjBkIiwibmFtZSI6InF0X3Z5YW1uM2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJtaW5fZGF0ZSJ9fX0seyJpZCI6InQwLnF0Xzc3cHF5M2gyMGQiLCJuYW1lIjoicXRfNzdwcXkzaDIwZCIsIm5hbWVzcGFjZSI6InQwIiwicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iOnsiZGF0YVRyYW5zZm9ybWF0aW9uIjp7InNvdXJjZUZpZWxkTmFtZSI6InN1cHBsaWVyX2NvZGUifX19LHsiaWQiOiJ0MC5xdF90cmdrMDNoMjBkIiwibmFtZSI6InF0X3RyZ2swM2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJwX3BhY2thZ2VfZnJpZW5kbHkifX19LHsiaWQiOiJ0MC5xdF9xdzJ1ODNoMjBkIiwibmFtZSI6InF0X3F3MnU4M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJwbGFubmVkX2ltcHMiLCJhZ2dyZWdhdGlvbiI6Nn19fSx7ImlkIjoidDAucXRfdXhwdWI0aDIwZCIsIm5hbWUiOiJxdF91eHB1YjRoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoidG90YWxfaW1wcmVzc2lvbnMiLCJhZ2dyZWdhdGlvbiI6Nn19fV0sImF0dHJpYnV0ZUNvbmZpZyI6eyJjb21wb25lbnRBdHRyaWJ1dGUiOnsiZGlzcGxheUNvbmZpZ1ZlcnNpb24iOjAsImRhdGFzb3VyY2VDb25maWdWZXJzaW9uIjoyLCJ0b3AiOjAsImxlZnQiOjAsIndpZHRoIjo4MjMsImhlaWdodCI6NTcwfX0sImNvbXBvbmVudElkIjoiX19WSVpfQ0hBUlRfSURfXyIsInR5cGUiOiJwaXZvdC10YWJsZSIsInByZXNldCI6ImRlZmF1bHQiLCJiZWhhdmlvciI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5Ijoib25EaW1lbnNpb25FeHBhbmRDb2xsYXBzZSIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOlt7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJhY3Rpb24iLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6ImV4cGFuZC1jb2xsYXBzZSJ9fSx7ImtleSI6ImlzQ29udHJvbCIsInZhbHVlIjp7ImJvb2xWYWx1ZSI6ZmFsc2V9fSx7ImtleSI6InJlc291cmNlIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InR5cGUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6ImNvbmNlcHRzIn19LHsia2V5IjoidmFsdWUiLCJ2YWx1ZSI6eyJhcnJheVZhbHVlIjp7InZhbHVlIjpbeyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiaWQiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InQwLnF0Xzc3cHF5M2gyMGQifX0seyJrZXkiOiJuYW1lIiwidmFsdWUiOnsic3RyVmFsdWUiOiJxdF83N3BxeTNoMjBkIn19LHsia2V5IjoibmFtZXNwYWNlIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6InF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InN1cHBsaWVyX2NvZGUifX1dfX19XX19fV19fSx7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJpZCIsInZhbHVlIjp7InN0clZhbHVlIjoidDAucXRfdHJnazAzaDIwZCJ9fSx7ImtleSI6Im5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InF0X3RyZ2swM2gyMGQifX0seyJrZXkiOiJuYW1lc3BhY2UiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InQwIn19LHsia2V5IjoicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiZGF0YVRyYW5zZm9ybWF0aW9uIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvdXJjZUZpZWxkTmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicF9wYWNrYWdlX2ZyaWVuZGx5In19XX19fV19fX1dfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOlt7ImJlaGF2aW9yVHlwZSI6Im9uRGltZW5zaW9uRXhwYW5kQ29sbGFwc2UiLCJkcmlsbGRvd25QYXJhbWV0ZXJWYWx1ZSI6eyJ2YWx1ZXMiOlt7ImluaXQiOnsiaWQiOiJ0MC5xdF83N3BxeTNoMjBkIiwibmFtZSI6InF0Xzc3cHF5M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJzdXBwbGllcl9jb2RlIn19fSwiZGlyZWN0aW9uIjoxLCJjdXIiOnsiaWQiOiJ0MC5xdF83N3BxeTNoMjBkIiwibmFtZSI6InF0Xzc3cHF5M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsImRpc3BsYXlOYW1lIjoic3VwcGxpZXJfY29kZSIsInNlbWFudGljIjpbMzJdLCJsb29rZXJQcm9wZXJ0aWVzIjp7ImRlc2NyaXB0aW9uIjoiIn0sInR5cGUiOjAsImRhdGFUeXBlIjowLCJpc0RlZmF1bHQiOmZhbHNlLCJkZWZhdWx0QWdncmVnYXRpb24iOjAsIm5ld0RlZmF1bHRBZ2dyZWdhdGlvbiI6MCwiaXNBdXRvRmllbGQiOnRydWUsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJzdXBwbGllcl9jb2RlIn19LCJncm91cCI6IkNoYXJ0IGZpZWxkcyJ9LCJkcmlsbFRvIjp7ImlkIjoidDAucXRfdHJnazAzaDIwZCIsIm5hbWUiOiJxdF90cmdrMDNoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJkaXNwbGF5TmFtZSI6InBfcGFja2FnZV9mcmllbmRseSIsInNlbWFudGljIjpbMzJdLCJsb29rZXJQcm9wZXJ0aWVzIjp7ImRlc2NyaXB0aW9uIjoiIn0sInR5cGUiOjAsImRhdGFUeXBlIjowLCJpc0RlZmF1bHQiOmZhbHNlLCJkZWZhdWx0QWdncmVnYXRpb24iOjAsIm5ld0RlZmF1bHRBZ2dyZWdhdGlvbiI6MCwiaXNBdXRvRmllbGQiOnRydWUsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJwX3BhY2thZ2VfZnJpZW5kbHkifX0sImdyb3VwIjoiQ2hhcnQgZmllbGRzIn0sInR5cGUiOjMsImhpZXJhcmNoeSI6W3siaWQiOiJ0MC5xdF83N3BxeTNoMjBkIiwibmFtZSI6InF0Xzc3cHF5M2gyMGQiLCJuYW1lc3BhY2UiOiJ0MCIsInF1ZXJ5VGltZVRyYW5zZm9ybWF0aW9uIjp7ImRhdGFUcmFuc2Zvcm1hdGlvbiI6eyJzb3VyY2VGaWVsZE5hbWUiOiJzdXBwbGllcl9jb2RlIn19fSx7ImlkIjoidDAucXRfdHJnazAzaDIwZCIsIm5hbWUiOiJxdF90cmdrMDNoMjBkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoicF9wYWNrYWdlX2ZyaWVuZGx5In19fV19XX19XSwidmVyc2lvbiI6MX0aEQoNc3VwcGxpZXJfY29kZRABGhUKEXRvdGFsX2ltcHJlc3Npb25zEAUaEAoMcGxhbm5lZF9pbXBzEAU=')

In [ ]:
%%bigquery --project looker-studio-pro-452620

SELECT
    platform,
    SUM(impressions) AS total_impressions,
    SUM(spend) AS total_spend
  FROM
    `looker-studio-pro-452620.repo_stg.stg__adif__social_crossplatform`
  WHERE
    EXTRACT(YEAR FROM date_day) = 2026
  GROUP BY
    platform


In [ ]:
# dataframe:
# uuid: CD0076B5-0648-401E-AF04-362E4C318669
# output_variable:
# config_str:

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='', uuid='CD0076B5-0648-401E-AF04-362E4C318669')